> ## SOLUTION / ANSWER KEY &mdash; Lab 7.7
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-07-validate-output.ipynb`](../lab-07-validate-output.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 7.7 &mdash; Validate Before You Act

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Check the structured record has its required fields
- Reject values outside the allowed set
- Catch an ungrounded reply (an invented date) before it goes out

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps are **offline &amp; deterministic** (pure Python stdlib); the agent-assembly labs reuse the **LangChain-shaped shim** from Module 6. Advanced labs end with an **optional** cell that runs the **real** library. You are building the **email-drafting agent** &mdash; the client's Lab 4.1.

**Reference:** [Module 7 slides &mdash; Reliability: validate, then act](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-07-07"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
The line between a demo and production is **validation** (deck slide 12): never trust the model
blindly. Check the record **parses**, the **required fields** are present, the values are in the
**allowed set**, and &mdash; crucially for a drafted reply &mdash; that it **invents no promises**
(the ETA in the reply must match the real order). If it fails validation, **don't act**.

In [ ]:
ALLOWED_INTENTS = {"refund", "delivery", "cancel", "other"}
good = {"order_id": 4471, "intent": "delivery", "reply": "...due Friday..."}
print("we will check records like:", good)

## Your Turn
Complete `validate`: collect problems for a missing id, a bad intent, and an ungrounded ETA.

In [ ]:
ALLOWED_INTENTS = {"refund", "delivery", "cancel", "other"}

def validate(rec, order):
    problems = []
    if rec.get("order_id") is None:
        problems.append("missing order_id")
    if rec.get("intent") not in ALLOWED_INTENTS:
        problems.append("bad intent")
    # no invented promise: the real ETA must actually appear in the reply text
    if order["eta"] not in rec.get("reply", ""):
        problems.append("ungrounded eta")
    return {"ok": len(problems) == 0, "problems": problems}

ORDER = {"id": "4471", "eta": "Friday"}
try:
    ok = validate({"order_id": 4471, "intent": "delivery", "reply": "due Friday"}, ORDER)
    bad = validate({"order_id": None, "intent": "sing", "reply": "due Monday"}, ORDER)
    print('valid  ->', ok)
    print('invalid->', bad)
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("a well-grounded record passes", lambda: validate({"order_id": 4471, "intent": "delivery", "reply": "due Friday"}, {"id": "4471", "eta": "Friday"})["ok"] is True)
expect_true("a missing order_id is caught", lambda: "missing order_id" in validate({"order_id": None, "intent": "delivery", "reply": "due Friday"}, {"id": "4471", "eta": "Friday"})["problems"])
expect_true("a bad intent is caught", lambda: "bad intent" in validate({"order_id": 1, "intent": "sing", "reply": "due Friday"}, {"id": "4471", "eta": "Friday"})["problems"])
expect_true("an invented date (wrong ETA) is caught", lambda: "ungrounded eta" in validate({"order_id": 1, "intent": "delivery", "reply": "due Monday"}, {"id": "4471", "eta": "Friday"})["problems"])
expect_true("ok is True only when there are no problems", lambda: validate({"order_id": 1, "intent": "delivery", "reply": "Friday"}, {"id": "4471", "eta": "Friday"})["ok"] and not validate({"order_id": None, "intent": "delivery", "reply": "Friday"}, {"id": "4471", "eta": "Friday"})["ok"])

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Validate parses, fields, ranges, and grounding BEFORE you act. An automation that acts on unchecked output is a liability; one that validates first is something you can trust to run.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>